# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Ellarh101/ML-starter/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

*Name your lane — or say 'freestyle' and describe your own question. One short paragraph: why this one?*

### My chosen lane: **Lane 3 — Structured Content Archetype Clustering**

I chose this lane because I want to learn what recurring "types" of pages exist in a content inventory, and how clustering can group pages that behave alike without needing a label. Most of my earlier work was prediction with a known target; here there is no target, so the value comes from inspecting the groups and deciding what to do with each one. The clusters will be built from structured metrics (impressions, clicks, CTR, position, sessions, age, word count), not from article text, so I will call them *performance archetypes*, never "semantic clusters". I treat clusters as a lens on the inventory, not as true labels.

*The cell below declares the lane in code and checks that the starter data really contains the metrics this lane needs.*

In [1]:
# Lane declaration + a check that the starter data really fits this lane
import numpy as np
import pandas as pd
from pathlib import Path

LANE = "Lane 3 - Structured Content Archetype Clustering"
print("MY LANE:", LANE)
print()

CSV_NAME = "content_refresh_anonymized.csv"
RAW_URL = "https://raw.githubusercontent.com/Ellarh101/ML-starter/main/data/raw/" + CSV_NAME


def load_starter():
    """Load the starter CSV from the repo folders, or from GitHub as a fallback (e.g. in Colab)."""
    for p in [Path("data/raw") / CSV_NAME, Path("../../data/raw") / CSV_NAME,
              Path("../data/raw") / CSV_NAME, Path(CSV_NAME)]:
        if p.exists():
            return pd.read_csv(p)
    return pd.read_csv(RAW_URL)  # if this fails, reuse the loading line from your starter notebook


df_raw = load_starter()
print(f"Starter dataset: {df_raw.shape[0]:,} rows x {df_raw.shape[1]} columns")

# The structured metrics I plan to cluster on
INTENDED = ["impressions_90d", "clicks_90d", "sessions_90d", "content_age_days",
            "ctr", "avg_position", "word_count", "engagement_rate", "scroll_rate"]
FEATURES = [c for c in INTENDED if c in df_raw.columns]
missing = [c for c in INTENDED if c not in df_raw.columns]
print("Metrics found for clustering:", FEATURES)
if missing:
    print("Planned but NOT in this CSV (ignored):", missing)

# Lane 3 uses metrics only: show that the non-numeric columns are ids / category labels
non_numeric = df_raw.select_dtypes(exclude="number").columns.tolist()
print("\nNon-numeric columns (should be ids or category labels, not article text) and how many distinct values:")
print(df_raw[non_numeric].nunique().to_string())

MY LANE: Lane 3 - Structured Content Archetype Clustering

Starter dataset: 30,000 rows x 44 columns
Metrics found for clustering: ['impressions_90d', 'clicks_90d', 'sessions_90d', 'content_age_days', 'ctr', 'avg_position', 'word_count', 'engagement_rate', 'scroll_rate']

Non-numeric columns (should be ids or category labels, not article text) and how many distinct values:
content_id           30000
client_id               32
competition_level        3
content_type             3
main_intent              4
provider_used            2
model_used               5
age_tier                 4
freshness_tier           4
word_count_tier          4
char_count_tier          4
impression_tier          4
position_tier            5
trend_direction          5


## 2. The question: decision, action, cost of a wrong call

*What decision does your work improve? Who acts on it? What does a wrong recommendation cost?*

- **Search question:** What recurring performance archetypes exist across the content inventory, and which review action (protect, improve, rewrite, merge, prune, monitor) fits each one?
- **Unit of analysis:** one page (one `content_id`) in the starter dataset, after the starter filters (impressions > 0, age of at least 90 days).
- **Output:** cluster profiles (typical numbers per cluster), a name for each archetype, and an action mapping from archetype to review action.
- **Decision and action:** A content team has limited review time. The output helps them decide which group of pages to look at first and what kind of work to do (for example protect strong pages, rewrite stale visible pages, review weak pages before any pruning). A person reviews pages in a cluster; nothing is changed automatically.
- **Cost of a wrong call:** If a page lands in the wrong archetype, a reviewer may spend time on the wrong pages (a false alarm), or a page that needed attention stays unnoticed (a miss). The costliest mistake is acting on "prune" or "merge" for a page that was actually fine, because traffic could be lost. For that reason, recommendations are "review" suggestions with a human check, and pruning is never suggested from the cluster label alone.
- **Why data/ML can help:** The inventory has many pages and several metrics that have to be read together, which is hard to do by hand. Clustering can group pages with similar profiles without a label, and it may show types I would not think to define in advance. It is not just "train a model": the real work is choosing and scaling features, choosing the number of clusters, checking that clusters are stable, profiling them honestly, and linking each one to an action.

*The cell below backs these answers with numbers from the starter data: the unit of analysis, review capacity, what is at stake, and why one metric is not enough.*

In [2]:
# Numbers that back the framing in Section 2

# Unit of analysis: one page = one content_id, after the starter filters
pages = (df_raw[(df_raw["impressions_90d"] > 0) & (df_raw["content_age_days"] >= 90)]
         .drop_duplicates("content_id"))
print(f"Unit of analysis: {len(pages):,} pages (unique content_id = {pages['content_id'].nunique():,})")

# Decision: a reviewer has limited capacity
for cap in (20, 50):
    print(f"Capacity: a reviewer checking {cap} pages covers only {cap / len(pages):.2%} of the inventory")

# Cost of a wrong call: how much visibility sits in the pages where a miss would hurt
total_imp = pages["impressions_90d"].sum()
down = pages["trend_direction"].eq("down")
print(f"Stakes (miss): pages trending down are {down.mean():.1%} of pages and hold "
      f"{pages.loc[down, 'impressions_90d'].sum() / total_imp:.1%} of impressions")
top10_imp = pages["impressions_90d"].nlargest(int(len(pages) * 0.10)).sum()
print(f"Stakes (false alarm): the top 10% of pages hold {top10_imp / total_imp:.1%} of impressions, "
      "so wrongly pruning or merging a top page is the costly mistake")

# Why data/ML can help: no single metric tells the whole story
corr = pages[FEATURES].corr(method="spearman").abs()
off_diag = corr.where(~np.eye(len(corr), dtype=bool)).stack()
print(f"Why not one rule: {len(FEATURES)} metrics must be read together; "
      f"median pairwise |Spearman correlation| = {off_diag.median():.2f}")

Unit of analysis: 30,000 pages (unique content_id = 30,000)
Capacity: a reviewer checking 20 pages covers only 0.07% of the inventory
Capacity: a reviewer checking 50 pages covers only 0.17% of the inventory
Stakes (miss): pages trending down are 54.2% of pages and hold 51.3% of impressions
Stakes (false alarm): the top 10% of pages hold 70.2% of impressions, so wrongly pruning or merging a top page is the costly mistake
Why not one rule: 9 metrics must be read together; median pairwise |Spearman correlation| = 0.14


## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*

In [3]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

# Number 1: size of the sample
print(f"1) Pages: {len(pages):,} from {pages['client_id'].nunique()} clients")

# Number 2: concentration of visibility (do a few pages dominate?)
imp = pages["impressions_90d"].sort_values(ascending=False)
top10_share = imp.head(int(len(imp) * 0.10)).sum() / imp.sum()
print(f"2) Top 10% of pages hold {top10_share:.1%} of 90-day impressions "
      f"(median {imp.median():,.0f} vs mean {imp.mean():,.0f})")

# Number 3: is there group structure? KMeans + silhouette on scaled structured metrics
X = pages[FEATURES].fillna(0).copy()
for col in X.columns:
    if X[col].min() >= 0 and X[col].skew() > 2:   # log-scale heavily skewed counts
        X[col] = np.log1p(X[col])
Xs = StandardScaler().fit_transform(X)
sample = np.random.RandomState(42).choice(len(Xs), size=min(5000, len(Xs)), replace=False)
scores = {}
for k in range(2, 7):
    labels = KMeans(n_clusters=k, n_init=10, random_state=42).fit_predict(Xs)
    scores[k] = silhouette_score(Xs[sample], labels[sample])
    print(f"   k={k}: silhouette = {scores[k]:.3f}")
best_k = max(scores, key=scores.get)
print(f"3) Best silhouette: k={best_k} ({scores[best_k]:.3f}); values near 0 mean groups overlap, near 1 mean well separated")

1) Pages: 30,000 from 32 clients
2) Top 10% of pages hold 70.2% of 90-day impressions (median 731 vs mean 5,200)
   k=2: silhouette = 0.243
   k=3: silhouette = 0.211
   k=4: silhouette = 0.227
   k=5: silhouette = 0.220
   k=6: silhouette = 0.225
3) Best silhouette: k=2 (0.243); values near 0 mean groups overlap, near 1 mean well separated


**What these numbers say (fill in after running, using your real output):**

- Number 1 (size): *one sentence on how many pages and clients, and whether that is enough to find groups.*
- Number 2 (concentration): *one sentence, for example "the top 10% of pages hold X% of impressions, so pages behave very differently and one average would hide that."*
- Number 3 (group structure): *one sentence on which k scored highest and what that value means. A modest score suggests the groups overlap, so I will treat the archetypes as a rough lens, not sharp categories.*

Together these suggest the inventory is uneven and may contain distinct page types, which makes this lane worth the next 7 weeks. This is a first look, not a finding.

## 4. Careful words: what I can and can't claim

*Write what your work will be able to say (observed, directional, decision-support) — and what it never will (causal proof, 'predicting Google').*

**What this work can say (observed, directional, decision-support):**
- In this starter sample, pages fall into groups with similar metric profiles, and the groups differ in measurable ways (shown in the cluster profiles).
- A cluster suggests which pages may deserve review first. It supports a human decision.
- Differences between clusters are associated with the metrics I used; they describe what was observed in this sample.

**What it cannot say:**
- It cannot prove that a refresh, rewrite, merge or prune would cause a page to recover. That needs an experiment or a causal design.
- It cannot say anything about Google's algorithm, AI citations or AI rankings.
- It cannot claim "semantic" grouping, because there is no article text in the data.
- Clusters are not true labels. Changing the features, the scaling or the number of clusters could give different groups, so I will check stability before trusting any archetype name.
- Results come from a 30,000-row anonymized starter slice, not the full warehouse, so they may not hold elsewhere.

*The cell below checks two of these limits in code: the scope of the data, and how stable the clusters are.*

In [4]:
from sklearn.metrics import adjusted_rand_score

# Limit 1: scope. This is a starter slice, not the full warehouse
print(f"Scope: {len(df_raw):,} rows from {df_raw['client_id'].nunique()} clients (a starter slice, not the ~79M-row warehouse)")

# Limit 2: not "semantic". Every clustering feature must be a numeric metric
assert all(pd.api.types.is_numeric_dtype(df_raw[c]) for c in FEATURES), "A non-numeric feature slipped in"
print(f"Check passed: all {len(FEATURES)} clustering features are numeric metrics, so groups reflect metrics, not meaning")

# Limit 3: clusters are a lens, not true labels. Do two random starts agree?
a = KMeans(n_clusters=best_k, n_init=10, random_state=1).fit_predict(Xs)
b = KMeans(n_clusters=best_k, n_init=10, random_state=2).fit_predict(Xs)
print(f"Stability (k={best_k}): adjusted Rand index between two random starts = {adjusted_rand_score(a, b):.2f} "
      "(1.00 = identical groups; much lower = the archetypes are shaky)")

Scope: 30,000 rows from 32 clients (a starter slice, not the ~79M-row warehouse)
Check passed: all 9 clustering features are numeric metrics, so groups reflect metrics, not meaning
Stability (k=2): adjusted Rand index between two random starts = 1.00 (1.00 = identical groups; much lower = the archetypes are shaky)


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.